# Word embeddings and sentence embeddings

You turn sentences into rows of numbers in three ways: TF-IDF, word embeddings, and sentence embeddings. You compare them on four sentences first, then on the AITA replies.

Work with a partner. Fill in the cells that start with a comment and use the check comments to confirm your result. Make a copy named `week09_live_workspace_mine.ipynb`.

In [ ]:
# Install sentence-transformers, gensim, and the other optional NLP packages into the course environment.
# Run this once on your own computer. It can take a few minutes. Skip it on Colab, which already has them.
!uv sync --frozen --group optional-nlp

In [ ]:
import re
import pandas as pd
import gensim.downloader
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
from sklearn.decomposition import PCA
import seaborn as sns

pd.set_option("display.max_colwidth", 300)  # show more of each reply

## 1. Word embeddings

In [ ]:
# GloVe: word embeddings trained on Wikipedia and news, with 50 numbers per word.
# The first run downloads about 66 MB.
glove = gensim.downloader.load("glove-wiki-gigaword-50")

In [ ]:
glove["wife"]  # the 50 numbers for "wife"

In [ ]:
glove.most_similar("wife")  # the words with the closest rows. Check: daughter 0.96, husband 0.95

In [ ]:
# Find the closest words to "housework". Check: chores first, 0.79

Are the closest words to *wife* synonyms? What do they have in common?

## 2. Compare four sentences three ways

Run these cells and read the scores. Part 3 uses the same code on the AITA replies.

In [ ]:
query = "You are wrong."
examples = pd.DataFrame({"text": [
    "You are not wrong.",
    "You did nothing wrong.",
    "You are mistaken.",
    "The cat sat on the mat.",
]})

In [ ]:
# TF-IDF gives each sentence one number per word: high for words common in it and rare in the others
tfidf = TfidfVectorizer(stop_words="english")
example_tfidf_rows = tfidf.fit_transform(examples["text"])  # fit learns the words; one row per sentence
query_tfidf_row = tfidf.transform([query])  # transform gives the query the same word columns

In [ ]:
tfidf_scores = cosine_similarity(query_tfidf_row, example_tfidf_rows)  # one row: the query against each sentence
examples["tfidf_score"] = tfidf_scores[0]  # [0] takes that row
examples.round(2)  # Check: 1.0, 0.62, 0.0, 0.0

In [ ]:
# One row for a sentence: the average of the rows for its words.
# Stop words and words GloVe does not know are left out.
def average_word_vector(text):
    words = re.findall(r"[a-z]+", text.lower())
    words = [w for w in words if w in glove and w not in ENGLISH_STOP_WORDS]
    return sum(glove[w] for w in words) / len(words)

In [ ]:
query_word_row = average_word_vector(query)  # one row of 50 numbers for the query
example_word_rows = [average_word_vector(text) for text in examples["text"]]  # one row for each sentence

In [ ]:
word_scores = cosine_similarity([query_word_row], example_word_rows)  # the [ ] makes the query's row a table with one row
examples["word_score"] = word_scores[0]
examples.round(2)  # Check: 1.0, 0.95, 0.71, 0.43

In [ ]:
# A sentence model reads a whole text, up to about 200 words, and gives one row of 384 numbers for it.
# The first run downloads about 90 MB.
model = SentenceTransformer("all-MiniLM-L6-v2")

In [ ]:
query_sentence_row = model.encode([query])  # the query in a list gives a table with one row
example_sentence_rows = model.encode(examples["text"].tolist())  # one row for each sentence

In [ ]:
sentence_scores = cosine_similarity(query_sentence_row, example_sentence_rows)
examples["sentence_score"] = sentence_scores[0]
examples.round(2)  # Check: 0.83, 0.39, 0.69, 0.01

Why does TF-IDF give "You are mistaken." a score of 0? Which method puts it above "You did nothing wrong."?

## 3. Search the AITA replies

Use the code from Part 2 on the replies, with TF-IDF and the sentence model.

In [ ]:
url = "https://raw.githubusercontent.com/macss-berkeley/compss-211a/main/docs/interactives/data/week08/aita_replies.csv"
comments = pd.read_csv(url)
comments["marker"] = comments["body"].str.upper().str.extract(r"^(NTA|YTA)\b")
replies = comments.dropna(subset=["marker"]).reset_index(drop=True)  # keep the replies that start with NTA or YTA
replies["marker"].value_counts()  # Check: NTA 657, YTA 378

In [ ]:
# Make reply_tfidf_rows: fit a new TfidfVectorizer on replies["body"] with stop_words="english" and min_df=5 (keep words found in five or more replies). Check: reply_tfidf_rows.shape is (1035, 773)

In [ ]:
# Make reply_sentence_rows with model.encode. This can take a minute. Check: reply_sentence_rows.shape is (1035, 384)

In [ ]:
query = "You should break up with him."

In [ ]:
# Make query_tfidf_row with the vectorizer you fit on the replies. Check: query_tfidf_row.shape is (1, 773)

In [ ]:
# Score query_tfidf_row against reply_tfidf_rows and call the result tfidf_scores. Check: tfidf_scores.shape is (1, 1035)

In [ ]:
replies["tfidf_score"] = tfidf_scores[0]  # [0] takes the one row of scores
replies.sort_values("tfidf_score", ascending=False).head(5)[["idstr", "tfidf_score", "body"]]  # Check: the second is t1_efw6iht

In [ ]:
# Make query_sentence_row with model.encode. Check: query_sentence_row.shape is (1, 384)

In [ ]:
# Score query_sentence_row against reply_sentence_rows and call the result sentence_scores. Check: sentence_scores.shape is (1, 1035)

In [ ]:
# Save sentence_scores[0] in replies["sentence_score"] and show the five highest. Check: the second is t1_egpmt9p

Which list has more replies about breaking up? Which of them do not contain the word *break*?

Set `query` to "He expects his wife to do all the housework." and run the cells from `query` down again. Why are the TF-IDF results about wives and not housework? (`min_df=5` keeps only words found in five or more replies.)

## 4. Plot the replies

PCA turns each row of numbers into two numbers, so each reply can be drawn as a point. The next two cells plot the sentence embeddings.

In [ ]:
pca = PCA(n_components=2)  # keep two numbers per reply
sentence_points = pca.fit_transform(reply_sentence_rows)  # one row of two numbers for each reply
sentence_points.shape  # Check: (1035, 2)

In [ ]:
replies["x"] = sentence_points[:, 0]  # the first number
replies["y"] = sentence_points[:, 1]  # the second number
sns.scatterplot(data=replies, x="x", y="y", hue="marker", s=10)  # each dot is one reply

In [ ]:
# Make tfidf_points from reply_tfidf_rows in the same way. Check: tfidf_points.shape is (1035, 2)

In [ ]:
# Plot tfidf_points in the same way, colored by marker

Both plots split the NTA and YTA replies. Every reply starts with NTA or YTA. Could that explain both plots?

## 5. Your own search

Write a query about another topic in the replies, such as money or a wedding. Show the five closest replies with TF-IDF and with the sentence model. Which finds more replies about your topic?

In [ ]:
# Your query and the five closest replies with each method

## For your project

The sentence model reads only about the first 200 words of each text and ignores the rest. For longer texts, such as speeches or reports, use TF-IDF rows, or split each text into paragraphs and embed those.

## If you have time: find texts like one you already have

The row of any text can be the query. In a project, this finds more texts like one you want to quote or label.

In [ ]:
example_row = reply_sentence_rows[280]  # reply 280: "Nta. Honeatly break up with him. You can do better"
replies["example_score"] = cosine_similarity([example_row], reply_sentence_rows)[0]

In [ ]:
replies.sort_values("example_score", ascending=False).head(6)[["idstr", "example_score", "body"]]  # the first is reply 280 itself. Check: the second is t1_egpmt9p

In [ ]:
# Pick another reply from today's tables (its row number is on the left) and show the five replies closest to it

Save the notebook, restart the kernel, and run every cell.